# Solar Rooftop Potential — OSM Building Footprint ApproachCompute solar thermal and PV potential for a region using **OSM building footprints** as the geometry layer. This is the recommended approach for broad-coverage analysis where LoD2 data are unavailable.> OSM building footprints serve as a proxy for roof-hosting area. A `footprint_to_roof` correction factor converts footprint area to effective roof area. This factor is calibrated against LoD2 benchmark regions (see the methodological roadmap). Results from OSM mode are screening-level estimates, not investment-grade assessments.>> **Two geometry levels**: LoD2 (benchmark) → OSM (fallback). See `02_LoD2_Approach.ipynb` for the high-fidelity route.---

## 1. Imports

In [ ]:
import atliteimport matplotlib.pyplot as pltimport numpy as npimport xarray as xrfrom src.config import load_configfrom src.data_loading import load_cop, load_cutout, load_demand, load_shapefrom src.osm_processing import (    clip_to_region,    download_osm_buildings,    rasterize_buildings,)from src.solar_thermal import compute_solar_thermalplt.rcParams.update({"font.size": 11})

## 2. Configuration

In [ ]:
import logginglogging.basicConfig(level=logging.INFO)cfg = Nonetry:    cfg = load_config("config.yaml")    print("Loaded config from config.yaml")except FileNotFoundError:    print("No config.yaml found — using cell defaults")

## 3. Load Region Boundary

In [ ]:
shapefile = "data/leeste3035.gpkg" if cfg is None else cfg.resolve_path(cfg.region.shapefile)SHAPE_INDEX_NAME = "Name"shape = load_shape(str(shapefile))print(f"Loaded region: {shape.index[0]}, area: {shape.to_crs(3035).area.sum()/1e6:.2f} km²")

## 4. Load Demand & COP Data

In [ ]:
demand_path = "data/Last_240222.csv" if cfg is None else cfg.resolve_path(cfg.data.demand)cop_path = "data/COP_WP.xlsx" if cfg is None else cfg.resolve_path(cfg.data.cop)load_dhw, load_sph = load_demand(str(demand_path))cop = load_cop(str(cop_path))print(f"DHW demand: {load_dhw.sum():.0f} MWh/yr")print(f"SH  demand: {load_sph.sum():.0f} MWh/yr")print(f"COP range:  {cop.min():.2f} – {cop.max():.2f} (mean: {cop.mean():.2f})")

## 5. ERA5 Weather Cutout

In [ ]:
year = 2014 if cfg is None else cfg.parameters.yearregion_name = "Leeste" if cfg is None else cfg.region.namecutout_path = f"data/era5-{year}-{region_name}.nc" if cfg is None else str(cfg.resolve_path(cfg.data.cutout))cutout = load_cutout(str(cutout_path), year=year)print(f"Cutout: {cutout}")

## 6. Download OSM Building FootprintsDownloads building footprints from OpenStreetMap via the `osmnx` library. The data are cached locally as a GeoPackage for reuse.

In [ ]:
place_name = "Leeste, Weyhe, Germany" if cfg is None else cfg.region.namecache = "data/osm_buildings.gpkg"osm_buildings = download_osm_buildings(    place_name,    cache_path=cache,)print(f"Downloaded {len(osm_buildings)} building features")

## 7. Clip to Region

In [ ]:
# Reproject to region CRS and cliposm_buildings = osm_buildings.to_crs(shape.crs)osm_buildings = clip_to_region(osm_buildings, shape)footprint_m2 = osm_buildings.geometry.area.sum()print(f"Clipped buildings: {len(osm_buildings)}")print(f"Total footprint area: {footprint_m2:.0f} m² = {footprint_m2/1e6:.4f} km²")

## 8. Rasterize Building FootprintsRasterize the OSM building polygons to a GeoTIFF at the configured resolution for use with atlite's `ExclusionContainer`.

In [ ]:
resolution = 5 if cfg is None else cfg.parameters.resolutionbounds = shape.to_crs(3035).total_boundsout_raster = "data/osm_raster.tif"raster_arr, transform = rasterize_buildings(    osm_buildings, bounds, resolution, out_raster,)pixel_count = (raster_arr == 1).sum()pixel_area_m2 = pixel_count * resolution ** 2print(f"Raster: {pixel_count} building pixels = {pixel_area_m2:.0f} m² at {resolution} m")

## 9. OSM → Roof Correction FactorThe OSM building footprint area is a proxy for roof-hosting potential. A geometry-side correction factor (`footprint_to_roof`) is calibrated to reproduce the same effective panel-covered area as the LoD2 approach with `cap_per_sqkm = 19`.For the LoD2 benchmark:$$\text{panel\_area} = \frac{\text{LoD2\_rasterised\_area} \times \text{cap\_per\_sqkm}}{p_{\text{pv\_density}}} = \frac{35{,}474\,\text{m}^2 \times 19\,\text{MW/km}^2}{200\,\text{W/m}^2} \approx 3{,}370\,\text{m}^2$$This is the *implicit* panel-covered area (9.5% of the rasterised building area). The OSM correction factor preserves the same panel-covered area from OSM building footprints:$$\text{footprint\_to\_roof} = \frac{3{,}370}{90{,}836} \approx 0.037$$For **Leeste**: `footprint_to_roof = 0.037` (3,370 m² panel-equivalent area / 90,836 m² OSM footprints).

In [ ]:
f2r = 1.0 if cfg is None else cfg.parameters.footprint_to_roofpanel_area_m2 = footprint_m2 * f2r  # implicit panel-covered area (MW-equivalent)print(f"Footprint-to-roof factor: {f2r}")print(f"Panel-equivalent area:    {panel_area_m2:.0f} m²")print(f"OSM footprints:           {footprint_m2:.0f} m²")

## 10. Availability MatrixUse the rasterized OSM footprints as an exclusion mask for atlite. The availability matrix `A` gives the fraction of each ERA5 grid cell covered by OSM buildings.

In [ ]:
buildings_excluder = atlite.ExclusionContainer(crs=shape.crs, resolution=resolution)buildings_excluder.add_raster(out_raster)A = cutout.availabilitymatrix(shape, buildings_excluder)print(f"Availability matrix shape: {A.shape}")print(f"Max availability per cell: {A.max():.4f}")

## 11. Capacity MatrixThe capacity matrix converts availability to installable power. Both PV and solar thermal share this matrix.For OSM mode, the effective capacity density derives directly from the geometry:$$C_{i,j} = A_{i,j} \times \text{Area}_{i,j} \times \text{footprint\_to\_roof} \times 200$$

In [ ]:
area_km2 = cutout.grid.set_index(["y", "x"]).to_crs(shape.crs).area / 1e6area_km2 = xr.DataArray(area_km2, dims=("spatial",))osm_cap_per_sqkm = f2r * 200  # MW/km² of OSM footprint → usable roofcapacity_matrix = A.stack(spatial=["y", "x"]) * area_km2 * osm_cap_per_sqkmtotal_cap = capacity_matrix.sum().valuesprint(f"Effective capacity density: {osm_cap_per_sqkm:.2f} MW/km²")print(f"Total installed capacity:   {total_cap:.2f} MW")

## 12. Solar Thermal PotentialUsing flat-plate collector models (see `src/solar_thermal.py` for details). OSM mode uses latitude-optimal orientation since roof planes are unknown.

In [ ]:
st_params = cfg.parameters if cfg is not None else Nonest_potential = compute_solar_thermal(    cutout, shape, capacity_matrix,    params=st_params,    orientation="latitude_optimal",)coverage_st = st_potential.sum() / load_dhw.sum() * 100print(f"Solar thermal covers {coverage_st:.1f}% of DHW demand")

## 13. PV Potential

In [ ]:
pv = cutout.pv(    panel=atlite.solarpanels.CdTe,    matrix=capacity_matrix,    orientation="latitude_optimal",    index=shape.index,)pv = pv.to_pandas()pv = pv[SHAPE_INDEX_NAME]coverage_pv = pv.sum() / load_sph.sum() * 100print(f"PV covers {coverage_pv:.1f}% of space heating demand (via HP)")

## 14. PV-Powered Heat Pump

In [ ]:
hp = pv * copcoverage_hp = hp.sum() / load_sph.sum() * 100print(f"HP covers {coverage_hp:.1f}% of space heating demand")

## 15. Time-Series Plots

### 15.1 Solar Thermal Potential

In [ ]:
st_potential.plot.line(    linewidth=0.5, figsize=(14, 4),    xlabel="Time", ylabel="Solar Thermal Potential (MW)",    title="Solar Thermal Potential — OSM Buildings")plt.show()

### 15.2 Domestic Hot Water Demand

In [ ]:
load_dhw.plot.line(    linewidth=0.5, figsize=(14, 4),    xlabel="Time", ylabel="DHW Demand (MW)",    title="Domestic Hot Water Demand")plt.show()

### 15.3 Solar Thermal — DHW Mismatch

In [ ]:
diff1 = st_potential - load_dhwdiff1.plot.area(    figsize=(14, 4), xlabel="Time",    ylabel="Mismatch (MW)",    title="Solar Thermal to DHW: Production minus Consumption")plt.show()diff1.resample("D").mean().plot.area(    figsize=(14, 4), xlabel="Time",    ylabel="Mismatch (MW)",    title="Daily Average Mismatch")plt.show()

### 15.4 PV Potential

In [ ]:
pv.plot.line(    linewidth=0.5, figsize=(14, 4),    xlabel="Time", ylabel="PV Potential (MW)",    title="PV Potential — OSM Buildings")plt.show()

### 15.5 Heat Pump Thermal Output

In [ ]:
hp.plot.line(    linewidth=0.5, figsize=(14, 4),    xlabel="Time", ylabel="HP Thermal Output (MW)",    title="PV-Powered Heat Pump Output")plt.show()

### 15.6 Space Heating Demand

In [ ]:
load_sph.plot.line(    linewidth=0.5, figsize=(14, 4),    xlabel="Time", ylabel="SH Demand (MW)",    title="Space Heating Demand")plt.show()

### 15.7 Heat Pump — Space Heating Mismatch

In [ ]:
diff2 = hp - load_sphdiff2.plot.area(    figsize=(14, 4), xlabel="Time",    ylabel="Mismatch (MW)",    title="HP to SH: Production minus Consumption")plt.show()diff2.resample("D").mean().plot.area(    figsize=(14, 4), xlabel="Time",    ylabel="Mismatch (MW)",    title="Daily Average Mismatch")plt.show()

## 16. Sensitivity Analysis ($\alpha$ Variation)The parameter $\alpha$ allocates the shared roof area between PV ($\alpha$) and solar thermal ($1-\alpha$).

In [ ]:
alpha_values = [0, 0.2, 0.4, 0.6, 0.8, 1] if cfg is None else cfg.parameters.alpha_valuesst_a, hp_a = {}, {}demand_cover_dw_a, demand_cover_sh_a = {}, {}for a in alpha_values:    pv_temp = pv * a    st_temp = st_potential * (1 - a)    hp_temp = pv_temp * cop    diff1_t = st_temp - load_dhw    diff2_t = hp_temp - load_sph    st_a[a] = diff1_t[diff1_t > 0].sum()    hp_a[a] = diff2_t[diff2_t > 0].sum()    demand_cover_dw_a[a] = st_temp.sum() / load_dhw.sum() * 100    demand_cover_sh_a[a] = hp_temp.sum() / load_sph.sum() * 100print("Overproduction DHW (MWh):", st_a)print("Overproduction SH  (MWh):", hp_a)print("DHW coverage       (%):", demand_cover_dw_a)print("SH  coverage       (%):", demand_cover_sh_a)

### 16.1 Load Coverage Plot

In [ ]:
v1 = list(demand_cover_dw_a.values())v2 = list(demand_cover_sh_a.values())fig, ax = plt.subplots(figsize=(7, 5))ax.fill_between(alpha_values, v1, alpha=0.5, label="Domestic Hot Water")ax.fill_between(alpha_values, v2, alpha=0.5, label="Space Heating")ax.set_xlabel("$\\alpha$")ax.set_ylabel("Load Coverage [%]")ax.set_title("Sensitivity Analysis — Load Coverage (OSM Buildings)")ax.legend(loc="upper left")ax.grid(True)plt.show()

### 16.2 Overproduction Plot

In [ ]:
v3 = np.array(list(st_a.values())) / load_dhw.sum() * 100v4 = np.array(list(hp_a.values())) / load_sph.sum() * 100 / cop.mean()fig, ax = plt.subplots(figsize=(7, 5))ax.fill_between(alpha_values, v3, alpha=0.5, label="Domestic Hot Water")ax.fill_between(alpha_values, v4, alpha=0.5, label="PV")ax.set_xlabel("$\\alpha$")ax.set_ylabel("Overproduction / Yearly Demand [%]")ax.set_title("Sensitivity Analysis — Overproduction (OSM Buildings)")ax.legend(loc="upper left")ax.grid(True)plt.show()

---**Note on interpretation**: These results are based on OSM building footprints with a `footprint_to_roof` geometry correction. For accurate, site-specific planning, use the LoD2 approach (see `02_LoD2_Approach.ipynb`). OSM results are suitable for early-stage regional screening.